In [2]:
import fitz
import pytesseract
from PIL import Image
import numpy as np
import cv2
import os
import unicodedata
import re

%cd /media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject

BASE_DIR = "Datasets/Unstructured_Data/8asl_amwal"

BIDI_RE = re.compile('[\u200E\u200F\u202A-\u202E\u2066-\u2069]')

def preprocess(img):
    img = np.array(img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.medianBlur(gray, 3)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(th)

def clean_text(text):
    text = BIDI_RE.sub('', text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'\u00A0', ' ', text)
    text = re.sub(r'[ ]{2,}', ' ', text)
    return text

# 🔥 Loop over all PDFs
for filename in os.listdir(BASE_DIR):
    
    if filename.lower().endswith(".pdf"):
        
        pdf_path = os.path.join(BASE_DIR, filename)
        output_txt = os.path.join(
            BASE_DIR,
            filename.replace(".pdf", "_ultra_clean.txt")
        )
        
        print(f"\n==============================")
        print(f"Processing file: {filename}")
        print(f"==============================")
        
        doc = fitz.open(pdf_path)
        all_text = ""
        
        for i, page in enumerate(doc):
            print(f"  Page {i+1}/{len(doc)}")
            
            mat = fitz.Matrix(4, 4)  # High DPI
            pix = page.get_pixmap(matrix=mat)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            
            img = preprocess(img)
            
            text = pytesseract.image_to_string(
                img,
                lang="ara",
                config="--psm 6"
            )
            
            text = clean_text(text)
            all_text += f"\n\n===== PAGE {i+1} =====\n\n"
            all_text += text
        
        with open(output_txt, "w", encoding="utf-8") as f:
            f.write(all_text)
        
        print(f"✔ Saved: {output_txt}")

print("\n🔥 All PDFs processed successfully.")

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject

Processing file: قانون-مكافحة-غسل-الامول.pdf
  Page 1/8
  Page 2/8
  Page 3/8
  Page 4/8
  Page 5/8
  Page 6/8
  Page 7/8
  Page 8/8
✔ Saved: Datasets/Unstructured_Data/8asl_amwal/قانون-مكافحة-غسل-الامول_ultra_clean.txt

🔥 All PDFs processed successfully.


1) CONTAINS
(Law)-[:CONTAINS]->(Article)
وصف: القانون يحتوي على مواد.
خصائص مقترحة: {section, note}
كارديناليتي: Law 1 — * Article

2) DEFINES
(Article)-[:DEFINES]->(Offense)
وصف: المادة تعرف/تحدد الفعل الجرمي.
خصائص مقترحة: {clause_ref, element_label} 
كارديناليتي: Article 1 — * Offense

3) INCURS
(Article)-[:INCURS]->(Penalty)
وصف: المادة تترتب عليها عقوبة/جزاء.
خصائص مقترحة: {severity,min_term,max_term,unit,notes}
كارديناليتي: Article 1 — * Penalty

4) RELATED_TO
(Offense)-[:RELATED_TO]->(Subject)
وصف: الربط الموضوعي (فئة/قسم قانوني).
خصائص مقترحة: {relevance_score}
كارديناليتي: Offense * — 1 Subject

5) USES_CONCEPT
(Article)-[:USES_CONCEPT]->(LegalConcept)
وصف: المادة تعتمد على مفهوم قانوني/مصطلح (مثل القصد، الشروع).
خصائص مقترحة: {role:'element'|'defense'|'condition'}
كارديناليتي: Article * — * LegalConcept

6) INTERPRETED_BY
(Article)-[:INTERPRETED_BY]->(JudicialPrinciple)
وصف: المادة فُسّرت بواسطة مبدأ نقضي/مبدأ قضائي.
خصائص مقترحة: {citation, judgment_id, interpretation_note}
كارديناليتي: Article * — * JudicialPrinciple

7) BASED_ON
(Article)-[:BASED_ON]->(ConstitutionalProvision)
وصف: المادة تستند إلى نص دستوري/مقتضى دستوري.
خصائص مقترحة: {const_ref, justification}
كارديناليتي: Article * — * ConstitutionalProvision

8) REFERENCES
(Article)-[:REFERENCES]->(Article)
وصف: إحالة/استشهاد بين مواد (إحالة، استبعاد، تنسيق نصي).
خصائص مقترحة: {ref_type:'cites'|'amends'|'see_also', clause}
كارديناليتي: Article * — * Article

9) AMENDED_BY
(Law)-[:AMENDED_BY]->(Law)
وصف: قانون معدل بواسطة قانون آخر.
خصائص مقترحة: {amend_date, amendment_note}
كارديناليتي: Law * — * Law

10) REQUIRES_ELEMENT
(Offense)-[:REQUIRES_ELEMENT]->(LegalConcept)
وصف: الجريمة تتطلب عنصرًا قانونيًا محددًا (مثل mens_rea أو actus_reus).
خصائص مقترحة: {element_type:'mens_rea'|'actus_reus'|'result'}
كارديناليتي: Offense * — * LegalConcept

11) AFFECTS_PENALTY
(LegalConcept)-[:AFFECTS_PENALTY]->(Penalty)
وصف: مفهوم/حالة قانونية تؤثر على قيمة/نوع العقوبة (مثلاً ظرف مخفف/مشدِّد إذا عزمه المبدأ).
خصائص مقترحة: {effect:'increase'|'decrease'|'exclude', magnitude}
كارديناليتي: LegalConcept * — * Penalty

12) CITES_PRINCIPLE
(JudicialPrinciple)-[:CITES]->(Article)
وصف: مبدأ نقضي يقتبس أو يستند إلى مادة محددة (علاقة عكسية مفيدة للبحث).
خصائص مقترحة: {case_ref}
كارديناليتي: JudicialPrinciple * — * Article

CREATE CONSTRAINT IF NOT EXISTS FOR (l:Law) REQUIRE l.law_id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (a:Article) REQUIRE a.article_id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (o:Offense) REQUIRE o.offense_id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (p:Penalty) REQUIRE p.penalty_id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (s:Subject) REQUIRE s.subject_id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (lc:LegalConcept) REQUIRE lc.concept_id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (cp:ConstitutionalProvision) REQUIRE cp.const_id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (jp:JudicialPrinciple) REQUIRE jp.principle_id IS UNIQUE;

{
  "law": { "name": "قانون العقوبات", "law_id": "" },
  "articles": [
    {
      "article_id": "",
      "number": "",
      "title": "",
      "full_text": "",
      "offenses": [
        { "name": "", "description": "" }
      ],
      "penalties": [
        { "type": "", "min": "", "max": "", "notes": "" }
      ],
      "legal_concepts": [],
      "subjects": [],
      "references": []
    }
  ]
}

{
  "law": { "name": "قانون الإجراءات الجنائية", "law_id": "" },
  "articles": [
    {
      "article_id": "",
      "number": "",
      "title": "",
      "full_text": "",
      "procedures": [
        { "type": "قبض/تفتيش/تحقيق/محاكمة/طعن", "conditions": "" }
      ],
      "legal_concepts": [],
      "subjects": [],
      "references": []
    }
  ]
}

{
  "law": { "name": "قانون مكافحة المخدرات", "law_id": "" },
  "classified_substances": [],
  "articles": [
    {
      "article_id": "",
      "number": "",
      "full_text": "",
      "offenses": [
        { "name": "إحراز/اتجار/تعاطي", "description": "" }
      ],
      "penalties": [
        { "type": "", "min": "", "max": "" }
      ],
      "aggravating_factors": [],
      "subjects": [],
      "references": []
    }
  ]
}

{
  "law": { "name": "قانون مكافحة المخدرات", "law_id": "" },
  "classified_substances": [],
  "articles": [
    {
      "article_id": "",
      "number": "",
      "full_text": "",
      "offenses": [
        { "name": "إحراز/اتجار/تعاطي", "description": "" }
      ],
      "penalties": [
        { "type": "", "min": "", "max": "" }
      ],
      "aggravating_factors": [],
      "subjects": [],
      "references": []
    }
  ]
}

{
  "law": { "name": "قانون مكافحة الإرهاب", "law_id": "" },
  "definitions": [],
  "articles": [
    {
      "article_id": "",
      "number": "",
      "full_text": "",
      "offenses": [],
      "penalties": [],
      "special_procedures": [],
      "legal_concepts": [],
      "subjects": [],
      "references": []
    }
  ]
}

{
  "law": { "name": "قانون مكافحة الجرائم الإلكترونية", "law_id": "" },
  "definitions": [],
  "articles": [
    {
      "article_id": "",
      "number": "",
      "full_text": "",
      "offenses": [],
      "technical_elements": [],
      "digital_evidence_rules": [],
      "penalties": [],
      "subjects": [],
      "references": []
    }
  ]
}

{
  "law": { "name": "قانون الأسلحة", "law_id": "" },
  "definitions": ["weapon_types"],
  "articles": [
    {
      "article_id": "",
      "number": "",
      "full_text": "",
      "offenses": [],
      "licensing_rules": [],
      "seizure_rules": [],
      "penalties": [],
      "subjects": [],
      "references": []
    }
  ]
}

{
  "law": { "name": "مواد الدستور الجنائية", "law_id": "" },
  "constitutional_articles": [
    {
      "const_id": "",
      "number": "",
      "full_text": "",
      "rights": [],
      "limitations": [],
      "linked_laws": []
    }
  ]
}

{
  "law": { "name": "مبادئ محكمة النقض الجنائي", "law_id": "" },
  "principles": [
    {
      "principle_id": "",
      "text": "",
      "summary": "",
      "linked_articles": [],
      "legal_concepts": [],
      "subjects": []
    }
  ]
}